###**MovieLens 20M Analytics using PySpark**

This project implements a PySpark-based data processing pipeline on the **MovieLens 20M dataset** to demonstrate scalable data engineering practices. The workflow includes **data modeling, fill-rate analysis, data cleaning, logical validation, and performance optimization**, executed entirely using PySpark.

The cleaned data is used to generate two business-focused analytics:

1. A **Producers Report** providing average ratings, total ratings, and distinct user counts per movie.

2. A **Directors Report** identifying distinct audience tags per movie for movies with a minimum review threshold.

3. A **Genre Market Gap Analysis** identifying underrepresented genres with high audience satisfaction but relatively low engagement.

4. A **Hidden Gems (Long-Tail) Analysis** highlighting high-quality movies that receive strong ratings despite limited audience visibility.

###**Environment Setup:**

PySpark and OpenJDK are installed and configured with explicit environment paths, followed by a clean initialization of the Spark session using an IPv4 bind address to ensure stable execution and Spark UI accessibility.

In [ ]:
!pip install pyspark -q
!apt-get update -qq
!apt-get install openjdk-11-jdk-headless -qq

import os
# 2. Set strict environment paths
# Correct JAVA_HOME to point to the installed Java 17 version
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PYSPARK_PYTHON"] = "python3"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python3"

# Verify Java installation
print("Verifying Java installation:")
!java -version

# 3. Initialize Spark Session with a "Spark Context Reset"
from pyspark.sql import SparkSession

# Stop any previous failed sessions
try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("MovieLens") \
    .config("spark.ui.port", "4050") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

print("✅ Spark Session Initialized with IPv4 bind address!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package openjdk-11-jre-headless:amd64.
(Reading database ... 117528 files and directories currently installed.)
Preparing to unpack .../openjdk-11-jre-headless_11.0.29+7-1ubuntu1~22.04_amd64.deb ...
Unpacking openjdk-11-jre-headless:amd64 (11.0.29+7-1ubuntu1~22.04) ...
Selecting previously unselected package openjdk-11-jdk-headless:amd64.
Preparing to unpack .../openjdk-11-jdk-headless_11.0.29+7-1ubuntu1~22.04_amd64.deb ...
Unpacking openjdk-11-jdk-headless:amd64 (11.0.29+7-1ubuntu1~22.04) ...
Setting up openjdk-11-jre-headless:amd64 (11.0.29+7-1ubuntu1~22.04) ...
update-alternatives: using /usr/lib/jvm/java-11-openjdk-amd64/bin/jjs to provide /usr/bin/jjs (jjs) in auto mode
update-alternatives: using /usr/lib/jvm/java-11-openjdk-amd64/bin/rmid to provide /usr/bin/rmid

###**Spark UI Exposure:**

ngrok is configured to securely expose the Spark Web UI running on a local port, enabling real-time monitoring of Spark jobs from an external browser during execution.

In [ ]:
!pip install pyngrok -q
from pyngrok import ngrok

# 1. SET THE TOKEN (Do not put this in a print statement)
# Use quotes around the token to make it a string
my_token = "36mTeI5fSlKrlSF0bvWT8KwVZ9M_7Z5CdSzorSgofKxKyfbUa"
ngrok.set_auth_token(my_token)

# 2. CREATE THE TUNNEL
# This tells ngrok to open a "tunnel" to the Spark UI on port 4050
connection = ngrok.connect(4050)

# 3. GET AND PRINT THE PUBLIC URL
# 'connection.public_url' is the actual web link (e.g., https://abcd.ngrok.io)
print(f"Monitor your Spark jobs at: {connection.public_url}")

Monitor your Spark jobs at: https://fanciless-joltless-eli.ngrok-free.dev


###**Data Loading:**

The MovieLens CSV files are loaded into PySpark DataFrames and standardized using the .df naming convention to ensure consistency and readability across the analytics pipeline.

In [ ]:
import os
from pyspark.sql import functions as F
# 1. Define the folder path
folder_path = "/content/movies_dataset"
# 2. Loop through files and create dataframes dynamically
dataframes = {}
for filename in os.listdir(folder_path):
    if filename.endswith(".csv"):
        # Create a clean variable name (e.g., 'movie.csv' becomes 'movie_df')
        df_name = filename.replace(".csv", "").replace("-", "_") + "_df"

        # Load the file into Spark
        file_path = os.path.join(folder_path, filename)
        dataframes[df_name] = spark.read.csv(file_path, header=True, inferSchema=True)

        print(f"Loaded: {filename} as {df_name}")

Loaded: movie.csv as movie_df
Loaded: rating.csv as rating_df
Loaded: genome_tags.csv as genome_tags_df
Loaded: genome_scores.csv as genome_scores_df
Loaded: tag.csv as tag_df
Loaded: link.csv as link_df


###**Data Preview:**
Displays a sample of the loaded movie_df DataFrame to verify successful data ingestion and schema correctness.

In [ ]:
# Run a simple show command to trigger a Spark job
dataframes['movie_df'].show(5)

+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows


###**Fill Rate Analysis:**

Computes column-wise data completeness by calculating the percentage of non-null (and non-empty for string fields) values for each DataFrame to assess overall data quality.

In [ ]:
from pyspark.sql.types import StringType

def get_fill_rate(df, name):
    total = df.count()
    print(f"\n--- Fill Rate: {name} ---")

    select_expressions = []
    for c in df.columns:
        col_datatype = df.schema[c].dataType

        condition = F.col(c).isNotNull()

        if isinstance(col_datatype, StringType):
            condition = condition & (F.col(c) != "")

        select_expressions.append((F.count(F.when(condition, c)) / total * 100).alias(c))

    df.select(select_expressions).show()


get_fill_rate(dataframes['movie_df'], "Movies")
get_fill_rate(dataframes['rating_df'], "Ratings")
get_fill_rate(dataframes['tag_df'], "Tag")
get_fill_rate(dataframes['link_df'], "Links")
get_fill_rate(dataframes['genome_scores_df'], "genome_scores")
get_fill_rate(dataframes['genome_tags_df'], "genome_tags")


--- Fill Rate: Movies ---
+-------+-----+------+
|movieId|title|genres|
+-------+-----+------+
|  100.0|100.0| 100.0|
+-------+-----+------+


--- Fill Rate: Ratings ---
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
| 100.0|  100.0| 100.0|    100.0|
+------+-------+------+---------+


--- Fill Rate: Tag ---
+------+-------+-----+-----------------+
|userId|movieId|  tag|        timestamp|
+------+-------+-----+-----------------+
| 100.0|  100.0|100.0|99.99935562027991|
+------+-------+-----+-----------------+


--- Fill Rate: Links ---
+-------+------+-----------------+
|movieId|imdbId|           tmdbId|
+-------+------+-----------------+
|  100.0| 100.0|99.07617860546961|
+-------+------+-----------------+


--- Fill Rate: genome_scores ---
+-------+-----+---------+
|movieId|tagId|relevance|
+-------+-----+---------+
|  100.0|100.0|    100.0|
+-------+-----+---------+


--- Fill Rate: genome_tags ---
+-----+-----+
|tagId|  tag|
+

###**Data Cleaning and Logical Validation:**

Applies domain-specific rules to remove nulls, duplicates, invalid ranges, and logically incorrect records across all MovieLens tables, resulting in consistent and reliable cleaned DataFrames.

In [ ]:
from pyspark.sql import functions as F

# Store cleaned dataframes in a new dictionary
cleaned_dfs = {}

# 1. Clean Ratings (d): Must be 0.5-5.0, timestamp > 0, remove duplicates
cleaned_dfs['rating_df'] = dataframes['rating_df'].na.drop() \
    .filter((F.col("rating") >= 0.5) & (F.col("rating") <= 5.0)) \
    .filter(F.unix_timestamp(F.col("timestamp")) > 0) \
    .dropDuplicates(["userId", "movieId"])

# 2. Clean Movies: Drop nulls in ID/Title, trim whitespace
cleaned_dfs['movie_df'] = dataframes['movie_df'].na.drop(subset=["movieId", "title"]) \
    .withColumn("title", F.trim(F.col("title"))) \
    .dropDuplicates(["movieId"])

# 3. Clean Tags: Remove empty tags and nulls
cleaned_dfs['tag_df'] = dataframes['tag_df'].na.drop() \
    .filter(F.col("tag") != "") \
    .withColumn("tag", F.trim(F.col("tag")))

# 4. Clean Links: Ensure IDs are present
cleaned_dfs['link_df'] = dataframes['link_df'].na.drop(subset=["movieId"]) \
    .dropDuplicates(["movieId"])

# 5. Clean Genome Tags: Ensure unique tag IDs
cleaned_dfs['genome_tags_df'] = dataframes['genome_tags_df'].na.drop() \
    .dropDuplicates(["tagId"])

# 6. Clean Genome Scores: Relevance must be between 0 and 1
cleaned_dfs['genome_scores_df'] = dataframes['genome_scores_df'].na.drop() \
    .filter((F.col("relevance") >= 0.0) & (F.col("relevance") <= 1.0)) \
    .dropDuplicates(["movieId", "tagId"])

print("✅ All 6 files have been cleaned and logically verified.")

✅ All 6 files have been cleaned and logically verified.


###**Global Data Cleanliness Assessment:**

Calculates the overall data cleanliness percentage by comparing total records before and after cleaning across all datasets, providing a consolidated data quality metric.

In [ ]:
def calculate_global_cleanliness(original_dict, cleaned_dict):
    total_original = 0
    total_cleaned = 0

    for key in original_dict.keys():
        total_original += original_dict[key].count()
        total_cleaned += cleaned_dict[key].count()

    percentage = (total_cleaned / total_original) * 100
    print(f"Total Source Records: {total_original}")
    print(f"Total Cleaned Records: {total_cleaned}")
    print(f"Global Data Cleanliness Percentage: {percentage:.2f}%")

calculate_global_cleanliness(dataframes, cleaned_dfs)

Total Source Records: 21320455
Total Cleaned Records: 21320452
Global Data Cleanliness Percentage: 100.00%


###**Producers’ Report**

To generate the **Producers’ Movie Summary Report**, we started by using the **cleaned ratings dataset** to compute movie-level performance metrics. The ratings data was **grouped by `movieId`**, and for each movie we calculated three key measures:

1. **The average rating**, which represents overall audience satisfaction. This value was rounded to two decimal places for clarity.

2. **The total number of ratings**, which indicates how frequently the movie has been rated and serves as a measure of popularity.

3. **The distinct number of users** who rated the movie, which reflects the breadth of audience engagement.

Once these metrics were computed, the aggregated results were cached to optimize performance, as they are reused in subsequent operations.

Next, the aggregated rating statistics were joined with the cleaned movie metadata using `movieId` as the key. A broadcast join was applied because the `movie dataset` is relatively small compared to the `ratings dataset`, improving join efficiency.

Finally, the report was formatted to include the movie title along with the calculated metrics and **sorted in descending order of the number of ratings**, allowing producers to quickly identify the most popular and widely rated movies.

In [ ]:
from pyspark.sql import functions as F

# Step 1: Aggregate rating metrics (rounding the average here for precision)
movie_stats = cleaned_dfs['rating_df'].groupBy("movieId").agg(
    F.round(F.avg("rating"), 2).alias("average_rating"),
    F.count("rating").alias("number_of_ratings"),
    F.countDistinct("userId").alias("distinct_users_rated")
).cache()

# Step 2: Join with movie titles and format the output column
producer_report = cleaned_dfs['movie_df'].join(F.broadcast(movie_stats), "movieId", "left") \
    .select("title", "average_rating", "number_of_ratings", "distinct_users_rated") \
    .orderBy(F.desc("number_of_ratings"))

print("--- Requirement (1): Producer's Movie Summary Report ---")
# The column 'average_rating' now holds a rounded value, so show() displays it correctly
producer_report.show(10, truncate=False)


--- Requirement (1): Producer's Movie Summary Report ---
+-----------------------------------------+--------------+-----------------+--------------------+
|title                                    |average_rating|number_of_ratings|distinct_users_rated|
+-----------------------------------------+--------------+-----------------+--------------------+
|Pulp Fiction (1994)                      |4.18          |30415            |30415               |
|Forrest Gump (1994)                      |4.03          |30050            |30050               |
|Shawshank Redemption, The (1994)         |4.45          |28759            |28759               |
|Silence of the Lambs, The (1991)         |4.18          |28630            |28630               |
|Jurassic Park (1993)                     |3.67          |27234            |27234               |
|Star Wars: Episode IV - A New Hope (1977)|4.2           |24778            |24778               |
|Braveheart (1995)                        |4.05          |243

###**Directors’ Tag Analysis**

This code generates the Directors’ Report by analyzing audience-generated tags for movies with sufficient user feedback. Movies with **at least 10 ratings** are first identified from the aggregated ratings data and joined with the **`movies table` using `movieId`** to retrieve movie titles.

The filtered movies are then joined with the **tags table using `movieId` as the common key**, linking each movie to its user-generated tags. The resulting dataset is **grouped by `movieId` and `title`**, and the **count of distinct tag values** is computed to measure diversity in audience interpretation rather than tagging volume.

Finally, the results are **sorted in descending order of distinct tag count**, producing a ranked Directors’ Report based on thematic richness and audience perception.

In [ ]:
# Step 1: Identify movies with at least 10 reviews and get their titles
popular_movies = movie_stats.filter(F.col("number_of_ratings") >= 10) \
                           .join(cleaned_dfs['movie_df'], "movieId", "inner") \
                           .select("movieId", "title")

# Step 2: Join with tags and count distinct tags per movie
director_report = cleaned_dfs['tag_df'].join(popular_movies, "movieId", "inner") \
    .groupBy("movieId", "title") \
    .agg(F.countDistinct("tag").alias("distinct_tags_count")) \
    .select("title", "distinct_tags_count") \
    .orderBy(F.desc("distinct_tags_count"))

print("--- Requirement (2): Director's Tag Report (Min 10 Reviews) ---")
director_report.show(10, truncate=False)

--- Requirement (2): Director's Tag Report (Min 10 Reviews) ---
+---------------------------------------------------------+-------------------+
|title                                                    |distinct_tags_count|
+---------------------------------------------------------+-------------------+
|Pulp Fiction (1994)                                      |340                |
|Forrest Gump (1994)                                      |292                |
|Fight Club (1999)                                        |223                |
|Matrix, The (1999)                                       |215                |
|Avatar (2009)                                            |197                |
|Blade Runner (1982)                                      |196                |
|Star Wars: Episode IV - A New Hope (1977)                |189                |
|Shawshank Redemption, The (1994)                         |187                |
|Silence of the Lambs, The (1991)                       

###**Genre Market Gap Analysis**

This code performs a genre-level market analysis to evaluate how different movie genres perform in terms of audience satisfaction and engagement. The **`genres` attribute from the movies table** is split and exploded so that each movie–genre combination is analyzed independently. This allows accurate aggregation at the genre level rather than treating genres as a single combined attribute.

The expanded dataset is **joined with aggregated movie statistics using `movieId`**, connecting each genre with its corresponding audience metrics. The analysis then groups data by genre and computes the total number of movies produced, average audience rating, and total audience engagement for each genre.

From a business perspective, this analysis helps identify **market gaps and growth opportunities**. Genres with high average audience ratings but relatively low engagement indicate **underexposed or underserved segments**, where targeted marketing, improved recommendation placement, or strategic content investment could yield higher returns. Conversely, genres with high engagement but lower ratings may indicate oversaturation or quality issues, guiding content optimization and acquisition strategies.

In [ ]:
from pyspark.sql import functions as F

# Step 1: Explode the pipe-separated genres into individual rows
# Step 2: Join with movie stats to see how each genre performs
genre_market_analysis = cleaned_dfs['movie_df'] \
    .withColumn("genre", F.explode(F.split(F.col("genres"), r"\|"))) \
    .join(movie_stats, "movieId") \
    .groupBy("genre") \
    .agg(
        F.count("movieId").alias("total_movies_produced"),
        F.round(F.avg("average_rating"), 2).alias("avg_audience_score"),
        F.sum("number_of_ratings").alias("total_audience_engagement")
    ) \
    .orderBy(F.desc("avg_audience_score"))

print("--- Requirement (3): Industry Insight - Genre Market Gap Analysis ---")
genre_market_analysis.show(truncate=False)

--- Requirement (3): Industry Insight - Genre Market Gap Analysis ---
+------------------+---------------------+------------------+-------------------------+
|genre             |total_movies_produced|avg_audience_score|total_audience_engagement|
+------------------+---------------------+------------------+-------------------------+
|Film-Noir         |270                  |3.45              |98033                    |
|Documentary       |1865                 |3.44              |111768                   |
|War               |1008                 |3.37              |476798                   |
|IMAX              |192                  |3.31              |224784                   |
|Drama             |11265                |3.29              |4022494                  |
|Animation         |892                  |3.25              |518331                   |
|Romance           |3501                 |3.22              |1726405                  |
|Musical           |877                  |3.21    

 ### **Hidden Gems and Long-Tail Niche Discovery**

This analysis identifies **high-quality movies with limited visibility**, commonly referred to as hidden gems. Using the aggregated movie-level statistics derived from the cleaned ratings data, movies are evaluated based on their average audience rating and total number of ratings. Titles that receive **strong average ratings but relatively low engagement** are categorized into the long-tail segment.

The analysis leverages **`movieId` as the linking attribute** to combine rating statistics with movie metadata, ensuring that each identified hidden gem can be mapped back to its title and genre information. By filtering and ranking movies based on quality-to-visibility balance, the pipeline isolates content that is well-received but under-discovered.

#### Business Impact

From a business perspective, hidden gems represent **untapped value within the content catalog**. These movies often lack exposure rather than quality, making them ideal candidates for:

* **Recommendation system boosting**, improving personalization without acquiring new content.
* **Targeted promotions and rediscovery campaigns**, increasing watch time and user retention.
* **Catalog optimization**, ensuring high-quality content is not overshadowed by highly marketed titles.

Identifying long-tail niches allows platforms to maximize return on existing content assets while delivering more diverse and satisfying experiences to users.


In [ ]:
from pyspark.sql import functions as F

# 1. Calculate the thresholds for "High Rating" and "Low Popularity"
# We define niche as movies with < 100 ratings but an average rating > 4.0
niche_threshold_count = 100
niche_threshold_rating = 4.0

# 2. Filter movies that meet the "Hidden Gem" criteria
# Using the cached movie_stats from previous steps for efficiency (Requirement g)
hidden_gems = movie_stats.filter(
    (F.col("number_of_ratings") < niche_threshold_count) &
    (F.col("average_rating") > niche_threshold_rating)
)

# 3. Join with titles and genres to provide a final report for the Content Team
# Removed F.broadcast hint to let Spark's optimizer choose the best join strategy
hidden_gems_report = hidden_gems.join(cleaned_dfs['movie_df'], "movieId") \
    .select(
        "title",
        "genres",
        F.round("average_rating", 2).alias("avg_rating"),
        "number_of_ratings"
    ) \
    .orderBy(F.desc("average_rating"), F.asc("number_of_ratings"))

print("--- Requirement (4): Hidden Gems Report (Long-Tail Niche Discovery) ---")
hidden_gems_report.show(15, truncate=False)

--- Requirement (4): Hidden Gems Report (Long-Tail Niche Discovery) ---
+-----------------------------------------------------------------------------------------------+-----------------------+----------+-----------------+
|title                                                                                          |genres                 |avg_rating|number_of_ratings|
+-----------------------------------------------------------------------------------------------+-----------------------+----------+-----------------+
|Why Has Bodhi-Dharma Left for the East?: A Zen Fable (Dharmaga tongjoguro kan kkadalgun) (1989)|Drama                  |5.0       |1                |
|Canyon Passage (1946)                                                                          |Action|Romance|Western |5.0       |1                |
|Isadora (1968)                                                                                 |Drama                  |5.0       |1                |
|I Could Go on Singing